# Prodigy_Diagnostics — R2.2 v2 (5 esperimenti REDESIGNED)

**Setup canonico**: BASELINE_BPTT_864p_PRE_EVENTPROP (CF_FSNN_Net 864p, lambda_sr=0.5).
- 10 ep × 100 step = 1000 step totali (al limite community minimum, vedi W6)
- highway-only cache, AdamW non testato (solo Prodigy isolation)
- single-seed (fase qualitativa, NON comparativa quantitativa)

**5 esperimenti REDESIGNED** (in base a PRODIGY_DEEP_STUDY.md parte 2):

| ID | Lever isolato | Domanda |
|---|---|---|
| P-A | Baseline (T30 Prodigy replica) | Conferma d frozen come hypothesis? |
| P-B | + betas=(0.9, 0.99) (W1) | Beta3 piu reattivo aiuta? |
| P-C | + d_coef=2.0 (W2) | Scaling diretto di d_hat aiuta? |
| P-D | + d0=1e-5 (V2, fix konstmish) | Bump iniziale risolve frozen? |
| P-E | CANONICAL KOHYA completo + cosine | Setup community 'in azione' funziona? |

**Differenza vs v1**: ora ogni esperimento isola UN lever (V2/W1/W2 della community wisdom), non gioca con safeguard.

**Output per ogni run**: training_log.csv per-epoca + training_batch_log.csv per-batch (1000 righe).
Plot finale: d(step), lr_eff(step), loss(step) sovrapposti.

**Reference**: `document/PRODIGY_DEEP_STUDY.md` (parte 1 math + parte 2 community wisdom).


In [ ]:
# Cell 1 -- ENV check (file + Prodigy API + train.py CLI tutti i nuovi flags)
import sys, os, subprocess

# Verifica file critici nel repo root
for f in ['train.py', 'core/network.py', 'config.py',
          'data/cache_1500_highway_cut0.0_ou0.0.pt']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# prodigyopt installed
from prodigyopt import Prodigy
import inspect
sig = inspect.signature(Prodigy.__init__)
for p in ['safeguard_warmup', 'growth_rate', 'd_coef', 'd0', 'betas', 'use_bias_correction', 'weight_decay']:
    assert p in sig.parameters, f'MISSING Prodigy param: {p}'
print('[OK] Prodigy v1.1.2 API: tutti i 7 param richiesti presenti')

# train.py supporta i nuovi CLI flags
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--prodigy_safeguard_warmup', '--prodigy_growth_rate', '--prodigy_d_coef',
             '--prodigy_betas', '--prodigy_use_bias_correction', '--prodigy_d0',
             '--prodigy_weight_decay']:
    assert flag in help_txt, f'MISSING train.py flag: {flag}'
    print(f'  [OK] {flag}')

# scheduler cosine_no_restart
assert 'cosine_no_restart' in help_txt, 'MISSING scheduler cosine_no_restart'
print('  [OK] scheduler cosine_no_restart')

print('\nENV check passed.')


In [ ]:
# Cell 2 -- 5 esperimenti REDESIGNED (lever isolati)
EXPERIMENTS = [
    ('P-A', 'BASELINE T30 replica (Prodigy lib defaults + nostro safeguard=ON)', {'lr': 1.0, 'prodigy_betas': '0.9,0.999', 'prodigy_d_coef': 1.0, 'prodigy_d0': 1e-06, 'prodigy_weight_decay': 0.0001, 'prodigy_use_bias_correction': 0, 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf', 'scheduler': 'none'},
     'Replica esatta delle nostre T30 Prodigy run con BASELINE_PRE_EVENTPROP arch. Predizione: d resta frozen a ~1e-3 (failure mode F2). Conferma la nostra diagnosi prima di testare i fix.'),
    ('P-B', 'P-A + betas=(0.9, 0.99) [isola W1]', {'lr': 1.0, 'prodigy_betas': '0.9,0.99', 'prodigy_d_coef': 1.0, 'prodigy_d0': 1e-06, 'prodigy_weight_decay': 0.0001, 'prodigy_use_bias_correction': 0, 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf', 'scheduler': 'none'},
     "Test isolato W1 (madman404 'dramatic improvement'). beta3=beta2**0.5: default 0.999 -> beta3=0.9995 (decay lento). 0.99 -> beta3=0.995 (10x piu reattivo). Predizione: d cresce piu velocemente di P-A. Quanto? Resta da vedere."),
    ('P-C', 'P-A + d_coef=2.0 [isola W2]', {'lr': 1.0, 'prodigy_betas': '0.9,0.999', 'prodigy_d_coef': 2.0, 'prodigy_d0': 1e-06, 'prodigy_weight_decay': 0.0001, 'prodigy_use_bias_correction': 0, 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf', 'scheduler': 'none'},
     "Test isolato W2 (community consensus kohya/OneTrainer/bdsqlsz). d_hat = d_coef * d_numerator / d_denom. Raddoppio diretto della stima. Predizione: d cresce ~2x rispetto a P-A, ma se P-A e' frozen perche' d_denom = inf (s ~ 0), allora d_coef non aiuta. Diagnostico chiaro."),
    ('P-D', 'P-A + d0=1e-5 [isola V2 -- FIX UFFICIALE]', {'lr': 1.0, 'prodigy_betas': '0.9,0.999', 'prodigy_d_coef': 1.0, 'prodigy_d0': 1e-05, 'prodigy_weight_decay': 0.0001, 'prodigy_use_bias_correction': 0, 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf', 'scheduler': 'none'},
     "Test isolato V2 (LoganBooker, confermato da konstmish in Issue #27). d0 da 1e-6 a 1e-5 = bump 10x della stima iniziale di D. Predizione: il piu impattante dei 3 lever singoli. Da P-A in cui dlr~1e-6 inizia => d updates trascurabili, a dlr~1e-5 in cui Prodigy 'vede' i grad."),
    ('P-E', 'SETUP CANONICAL KOHYA completo + cosine_no_restart', {'lr': 1.0, 'prodigy_betas': '0.9,0.99', 'prodigy_d_coef': 2.0, 'prodigy_d0': 1e-05, 'prodigy_weight_decay': 0.01, 'prodigy_use_bias_correction': 1, 'prodigy_safeguard_warmup': 1, 'prodigy_growth_rate': 'inf', 'scheduler': 'cosine_no_restart'},
     "Setup 'Prodigy is ALL YOU NEED' (bdsqlsz/kohya/OneTrainer consensus) + cosine T_max=epochs (V3 konstmish). Tutti i 5 lever attivi insieme. Predizione: il VERO benchmark Prodigy 'in azione'. Se NON converge bene qui, allora Prodigy davvero non si adatta al nostro CF_FSNN."),
]

print(f'{len(EXPERIMENTS)} esperimenti R2.2 v2:')
for exp_id, label, cfg, expected in EXPERIMENTS:
    print(f'\n[{exp_id}] {label}')
    diff_from_pa = {k: v for k,v in cfg.items() if k != 'scheduler'}
    print(f'  config:   {cfg}')
    print(f'  expected: {expected[:150]}{"" if len(expected)<=150 else "..."}')


In [ ]:
# Cell 3 -- RUN sweep 5 esperimenti (~10-15 min ciascuno Azure)
import subprocess, sys, time, os, shutil

SKIP_IF_EXISTS = True
RESULTS_DIR = 'results/Prodigy_Study'
os.makedirs(RESULTS_DIR, exist_ok=True)

def _build_cli(exp_id, cfg):
    return [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', '10', '--max_steps_per_epoch', '100',
        '--batch_size', '8', '--val_batch_size', '32', '--seq_len', '50',
        '--cf_hidden_size', '32', '--cf_rank', '8',
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', '0.5',
        '--scenario_mix', 'highway', '--cut_in_ratio', '0.0',
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--optimizer', 'prodigy',
        '--data_cache', 'data/cache_1500_highway_cut0.0_ou0.0.pt',
        '--lr', str(cfg['lr']),
        '--max_lr', str(cfg['lr']),
        '--scheduler', cfg['scheduler'],
        '--prodigy_betas', str(cfg['prodigy_betas']),
        '--prodigy_d_coef', str(cfg['prodigy_d_coef']),
        '--prodigy_d0', str(cfg['prodigy_d0']),
        '--prodigy_weight_decay', str(cfg['prodigy_weight_decay']),
        '--prodigy_use_bias_correction', str(cfg['prodigy_use_bias_correction']),
        '--prodigy_safeguard_warmup', str(cfg['prodigy_safeguard_warmup']),
        '--prodigy_growth_rate', str(cfg['prodigy_growth_rate']),
        '--tag', f'R2v2_{exp_id}']

run_results = []
t_start = time.time()
for i, (exp_id, label, cfg, expected) in enumerate(EXPERIMENTS, 1):
    tag = f'R2v2_{exp_id}'
    dst_dir = f'{RESULTS_DIR}/{tag}'
    if SKIP_IF_EXISTS and os.path.isfile(f'{dst_dir}/training_log.csv'):
        print(f'\n[{i}/{len(EXPERIMENTS)}] [SKIP_EXIST] {tag}')
        run_results.append((exp_id, 'skipped'))
        continue
    print(f'\n{'='*78}\n[{i}/{len(EXPERIMENTS)}] {tag}: {label}\n{'='*78}')
    t0 = time.time()
    r = subprocess.run(_build_cli(exp_id, cfg), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    src_ck = f'checkpoints/{tag}'
    if os.path.isdir(src_ck):
        if os.path.isdir(dst_dir): shutil.rmtree(dst_dir)
        shutil.move(src_ck, dst_dir)
    el_total = time.time() - t_start
    eta = (el_total / i) * (len(EXPERIMENTS) - i) / 60
    print(f'\n[{i}/{len(EXPERIMENTS)}] {tag} -> {status}  this={el/60:.1f}min  ETA={eta:.0f}min')
    run_results.append((exp_id, status))

print(f'\n{"="*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min')
for exp_id, status in run_results:
    print(f'  {exp_id}: {status}')


In [ ]:
# Cell 4 -- ANALISI per-batch: d(step), lr_eff(step), loss(step) -- 5 curve
import pandas as pd, matplotlib.pyplot as plt, os
import numpy as np

RESULTS_DIR = 'results/Prodigy_Study'
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
summary_rows = []
for (exp_id, label, cfg, expected), col in zip(EXPERIMENTS, colors):
    tag = f'R2v2_{exp_id}'
    batch_log = f'{RESULTS_DIR}/{tag}/training_batch_log.csv'
    if not os.path.isfile(batch_log):
        print(f'[SKIP plot] {tag}: batch log missing')
        continue
    df = pd.read_csv(batch_log)
    step = df.index.values
    if 'prodigy_d' in df.columns:
        axes[0].plot(step, df['prodigy_d'], label=f'{exp_id}: {label[:40]}', alpha=0.85, color=col)
        axes[1].plot(step, df['prodigy_lr_eff'], label=exp_id, alpha=0.85, color=col)
    loss_col = 'loss_total' if 'loss_total' in df.columns else df.select_dtypes(include='number').columns[2]
    axes[2].plot(step, df[loss_col], label=exp_id, alpha=0.75, color=col)

    d_final = float(df['prodigy_d'].iloc[-1]) if 'prodigy_d' in df.columns else float('nan')
    d_max = float(df['prodigy_d'].max()) if 'prodigy_d' in df.columns else float('nan')
    lr_eff_max = float(df['prodigy_lr_eff'].max()) if 'prodigy_lr_eff' in df.columns else float('nan')
    loss_final = float(df[loss_col].iloc[-1])
    summary_rows.append({'exp': exp_id, 'd_final': d_final, 'd_max': d_max,
                        'lr_eff_max': lr_eff_max, 'loss_final': loss_final})

axes[0].set_yscale('log')
axes[0].set_ylabel('prodigy_d (log)')
axes[0].set_title('Dinamica adattatore d per batch (target: cresce stably)')
axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8, loc='best')
axes[1].set_yscale('log')
axes[1].set_ylabel('lr_eff = d * lr * bias_corr (log)')
axes[1].set_title('LR effettivo per batch (target: ~1e-3..1e-2)')
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8, loc='best')
axes[2].set_ylabel('train loss_total')
axes[2].set_xlabel('batch step')
axes[2].set_title('Train loss per batch')
axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8, loc='best')
fig.suptitle('R2.2 v2: Prodigy lever isolation (5 esperimenti, per-batch dynamics)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
out = 'opt_plots/R2v2_prodigy_diagnostics.png'
os.makedirs('opt_plots', exist_ok=True)
fig.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'[saved] {out}')

summary_df = pd.DataFrame(summary_rows)
from IPython.display import display, Markdown
display(Markdown('## Summary per-batch dynamics'))
display(summary_df)


In [ ]:
# Cell 5 -- val_total per epoca (5 esperimenti, baseline F2 reference incluso)
import pandas as pd, matplotlib.pyplot as plt, os

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for exp_id, label, cfg, expected in EXPERIMENTS:
    log = f'{RESULTS_DIR}/R2v2_{exp_id}/training_log.csv'
    if not os.path.isfile(log): continue
    df = pd.read_csv(log)
    if 'val_total' in df.columns:
        ax.plot(df['epoch'], df['val_total'], 'o-', label=f'{exp_id}: {label[:40]}',
                linewidth=1.7, markersize=5)

# Reference: baseline F2 (BASELINE_PRE_EVENTPROP) val_total best e' 0.2262 @ep14/15
ax.axhline(0.2262, color='black', linestyle='--', linewidth=1, alpha=0.6,
           label='F2 baseline BPTT+AdamW reference (val_total=0.2262 @15ep)')

ax.set_xlabel('epoch'); ax.set_ylabel('val_total')
ax.set_title('R2.2 v2: val_total per epoca (5 setup Prodigy + F2 BPTT baseline)')
ax.grid(alpha=0.3); ax.legend(fontsize=8, loc='best')
plt.tight_layout()
fig.savefig('opt_plots/R2v2_val_total_per_epoch.png', dpi=120, bbox_inches='tight')
plt.show()
print('[saved] opt_plots/R2v2_val_total_per_epoch.png')

# Tabella riassuntiva
import math
from IPython.display import display, Markdown
rows = []
for exp_id, label, cfg, expected in EXPERIMENTS:
    log = f'{RESULTS_DIR}/R2v2_{exp_id}/training_log.csv'
    if not os.path.isfile(log): continue
    df = pd.read_csv(log)
    vts = df['val_total'].values
    bi = int(df['val_total'].idxmin())
    rows.append({
        'exp': exp_id,
        'val_total_best': float(vts[bi]),
        'best_ep': bi+1,
        'val_total_last': float(vts[-1]),
        'val_data_at_best': float(df['val_data'].iloc[bi]) if 'val_data' in df.columns else float('nan'),
        'spike_rate_last': float(df['spike_rate'].iloc[-1]) if 'spike_rate' in df.columns else float('nan'),
    })
display(Markdown('## Risultati val per epoca'))
display(pd.DataFrame(rows))
